In [2]:
import pandas as pd

In [3]:
file = '/home/maciej/PycharmProjects/NYacc/data/ny_collisions.csv'

In [4]:
df = pd.read_csv(file, dtype={'ZIP CODE': str}, parse_dates=['ACCIDENT DATE'], keep_default_na=False, na_values=[''])

In [5]:
df.dtypes

ACCIDENT DATE                    datetime64[us]
ACCIDENT TIME                               str
BOROUGH                                     str
ZIP CODE                                    str
LATITUDE                                float64
LONGITUDE                               float64
LOCATION                                    str
ON STREET NAME                              str
CROSS STREET NAME                           str
OFF STREET NAME                             str
NUMBER OF PERSONS INJURED               float64
NUMBER OF PERSONS KILLED                float64
NUMBER OF PEDESTRIANS INJURED             int64
NUMBER OF PEDESTRIANS KILLED              int64
NUMBER OF CYCLIST INJURED                 int64
NUMBER OF CYCLIST KILLED                  int64
NUMBER OF MOTORIST INJURED                int64
NUMBER OF MOTORIST KILLED                 int64
CONTRIBUTING FACTOR VEHICLE 1               str
CONTRIBUTING FACTOR VEHICLE 2               str
CONTRIBUTING FACTOR VEHICLE 3           

In [6]:
df.to_parquet('ny_collisions.parquet', engine='pyarrow', compression='snappy')

In [7]:
df = pd.read_parquet('ny_collisions.parquet')

In [8]:
df.dtypes

ACCIDENT DATE                    datetime64[us]
ACCIDENT TIME                               str
BOROUGH                                     str
ZIP CODE                                    str
LATITUDE                                float64
LONGITUDE                               float64
LOCATION                                    str
ON STREET NAME                              str
CROSS STREET NAME                           str
OFF STREET NAME                             str
NUMBER OF PERSONS INJURED               float64
NUMBER OF PERSONS KILLED                float64
NUMBER OF PEDESTRIANS INJURED             int64
NUMBER OF PEDESTRIANS KILLED              int64
NUMBER OF CYCLIST INJURED                 int64
NUMBER OF CYCLIST KILLED                  int64
NUMBER OF MOTORIST INJURED                int64
NUMBER OF MOTORIST KILLED                 int64
CONTRIBUTING FACTOR VEHICLE 1               str
CONTRIBUTING FACTOR VEHICLE 2               str
CONTRIBUTING FACTOR VEHICLE 3           

In [9]:
df.describe()

,ACCIDENT DATE,LATITUDE,LONGITUDE,NUMBER OF PERSONS INJURED,NUMBER OF PERSONS KILLED,NUMBER OF PEDESTRIANS INJURED,NUMBER OF PEDESTRIANS KILLED,NUMBER OF CYCLIST INJURED,NUMBER OF CYCLIST KILLED,NUMBER OF MOTORIST INJURED,NUMBER OF MOTORIST KILLED,COLLISION_ID
count,1612178,1.415893e+06,1.415893e+06,1.612161e+06,1.612145e+06,1.612178e+06,1.612178e+06,1.612178e+06,1.612178e+06,1.612178e+06,1.612178e+06,1.612178e+06
mean,2016-04-03 12:22:05.011010,4.068864e+01,-7.386657e+01,2.631363e-01,1.185998e-03,5.060483e-02,6.302034e-04,2.098590e-02,9.242156e-05,1.916854e-01,4.633483e-04,2.765946e+06
min,2012-07-01 00:00:00,0.000000e+00,-2.012371e+02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.300000e+01
25%,2014-06-15 00:00:00,4.066882e+01,-7.397746e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.014464e+06
50%,2016-04-10 00:00:00,4.072258e+01,-7.393002e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,3.422826e+06
75%,2018-02-21 00:00:00,4.076789e+01,-7.386727e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,3.854210e+06
max,2019-11-26 00:00:00,4.231832e+01,0.000000e+00,3.100000e+01,8.000000e+00,2.700000e+01,6.000000e+00,4.000000e+00,2.000000e+00,3.100000e+01,5.000000e+00,4.249104e+06
std,NaN,1.200403e+00,2.438640e+00,6.584832e-01,3.644859e-02,2.316972e-01,2.577872e-02,1.445222e-01,9.677481e-03,6.206497e-01,2.334547e-02,1.506373e+06


Sprawdzenie ilości "pustych" dzielnic

In [10]:
print(df.groupby('BOROUGH', dropna=False).size())

BOROUGH
BRONX            156906
BROOKLYN         349682
MANHATTAN        272131
QUEENS           299308
STATEN ISLAND     49526
NaN              484625
dtype: int64


uzupełnienie brakujących dzielnic z użyciem istniejących danych geolokalizacyjnych

In [11]:
from sklearn.neighbors import KNeighborsClassifier

ma_dzielnice = df[df['BOROUGH'].notna() & df['LATITUDE'].notna() & df['LONGITUDE'].notna()]
brak_dzielnicy = df[df['BOROUGH'].isna() & df['LATITUDE'].notna() & df['LONGITUDE'].notna()]

if not brak_dzielnicy.empty and not ma_dzielnice.empty:

    knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean')

    knn.fit(ma_dzielnice[['LATITUDE', 'LONGITUDE']], ma_dzielnice['BOROUGH'])

    uzupelnione_dzielnice = knn.predict(brak_dzielnicy[['LATITUDE', 'LONGITUDE']])

    df.loc[brak_dzielnicy.index, 'BOROUGH'] = uzupelnione_dzielnice
    print(f"Uzupełniono {len(brak_dzielnicy)} wierszy na podstawie współrzędnych.")
else:
    print("Brak wierszy do uzupełnienia.")

Uzupełniono 316377 wierszy na podstawie współrzędnych.


In [12]:
#df[df.isna().any(axis=1)]

In [13]:
print(df.groupby('BOROUGH', dropna=False).size())

BOROUGH
BRONX            203791
BROOKLYN         427001
MANHATTAN        330269
QUEENS           410775
STATEN ISLAND     72094
NaN              168248
dtype: int64


Uzupełnianie brakujących wartości

In [14]:
df['GEOM_MISSING'] = df['LATITUDE'].isna().astype(int)

injury_cols = [
    'NUMBER OF PERSONS INJURED', 'NUMBER OF PERSONS KILLED',
    'NUMBER OF PEDESTRIANS INJURED', 'NUMBER OF PEDESTRIANS KILLED',
    'NUMBER OF CYCLIST INJURED', 'NUMBER OF CYCLIST KILLED',
    'NUMBER OF MOTORIST INJURED', 'NUMBER OF MOTORIST KILLED'
]
df[injury_cols] = df[injury_cols].fillna(0).astype(int)

other_columns = [
    'BOROUGH', 'ZIP CODE',
    'CONTRIBUTING FACTOR VEHICLE 1', 'CONTRIBUTING FACTOR VEHICLE 2', 'CONTRIBUTING FACTOR VEHICLE 3', 'CONTRIBUTING FACTOR VEHICLE 4', 'CONTRIBUTING FACTOR VEHICLE 5',
    'VEHICLE TYPE CODE 1', 'VEHICLE TYPE CODE 2', 'VEHICLE TYPE CODE 3', 'VEHICLE TYPE CODE 4', 'VEHICLE TYPE CODE 5',
    'ON STREET NAME', 'CROSS STREET NAME', 'OFF STREET NAME'

]

df[other_columns] = df[other_columns].fillna('UnKnOwN')

In [15]:
#df.to_csv('ny_collisions_updated.csv')

In [16]:
df.dtypes

ACCIDENT DATE                    datetime64[us]
ACCIDENT TIME                               str
BOROUGH                                     str
ZIP CODE                                    str
LATITUDE                                float64
LONGITUDE                               float64
LOCATION                                    str
ON STREET NAME                              str
CROSS STREET NAME                           str
OFF STREET NAME                             str
NUMBER OF PERSONS INJURED                 int64
NUMBER OF PERSONS KILLED                  int64
NUMBER OF PEDESTRIANS INJURED             int64
NUMBER OF PEDESTRIANS KILLED              int64
NUMBER OF CYCLIST INJURED                 int64
NUMBER OF CYCLIST KILLED                  int64
NUMBER OF MOTORIST INJURED                int64
NUMBER OF MOTORIST KILLED                 int64
CONTRIBUTING FACTOR VEHICLE 1               str
CONTRIBUTING FACTOR VEHICLE 2               str
CONTRIBUTING FACTOR VEHICLE 3           

3.2. Określenie najniebezpieczniejszych czynników wypadków w każdej z dzielnic NY
(suma zabitych i rannych; TOP 5 przyczyn)

In [17]:
df['TOTAL_VICTIMS'] = df['NUMBER OF PERSONS INJURED'] + df['NUMBER OF PERSONS KILLED']

wypadki_ofiary = df.groupby(['BOROUGH', 'CONTRIBUTING FACTOR VEHICLE 1'])['TOTAL_VICTIMS'].sum().reset_index(
    name='suma_ofiar')

wypadki_ofiary = wypadki_ofiary[wypadki_ofiary['BOROUGH'] != 'UnKnOwN']
#wypadki_ofiary = wypadki_ofiary[wypadki_ofiary['CONTRIBUTING FACTOR VEHICLE 1'] != 'Unspecified']

wypadki_ofiary = wypadki_ofiary.sort_values(by=['BOROUGH', 'suma_ofiar'], ascending=[True, False])

top5_ofiary = wypadki_ofiary.groupby('BOROUGH').head(5)
pivot_df = top5_ofiary.pivot(
    index='BOROUGH',
    columns='CONTRIBUTING FACTOR VEHICLE 1',
    values='suma_ofiar'
).fillna(0).astype(int)

sort_columns = pivot_df.sum(axis=0).sort_values(ascending=False).index

pivot_df = pivot_df[sort_columns]

# noinspection PyTypeChecker
pivot_style = (
    pivot_df.style
    .background_gradient(cmap='YlOrRd', axis=None)
    .set_caption("Suma ofiar wypadków w podziale na dzielnice i czynniki")
    .set_properties(**{
        'text-align': 'center',
        'font-family': 'Arial',
        'border': '1px solid #dee2e6'
    })
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#343a40'), ('color', 'white'), ('font-weight', 'bold')]},
        {'selector': 'th.row_heading', 'props': [('text-align', 'left'), ('background-color', '#f8f9fa'), ('color', 'black')]}
    ])
)

pivot_style

CONTRIBUTING FACTOR VEHICLE 1,Unspecified,Driver Inattention/Distraction,Failure to Yield Right-of-Way,Following Too Closely,Traffic Control Disregarded,Other Vehicular
BOROUGH,,,,,,
BRONX,21259,10374,4664,3779,0,2451
BROOKLYN,48043,21922,12593,6362,4868,0
MANHATTAN,17683,13680,5291,3097,0,2452
QUEENS,33112,24011,12761,8027,4597,0
STATEN ISLAND,5247,4422,1794,1122,615,0


3.3. Zaprezentowanie, ile zgonów oraz obrażeń zostało spowodowanych przez szybką jazdę w danej
dzielnicy

In [18]:
speeding = df[
    (df['CONTRIBUTING FACTOR VEHICLE 1'] == 'Unsafe Speed') |
    (df['CONTRIBUTING FACTOR VEHICLE 2'] == 'Unsafe Speed') |
    (df['CONTRIBUTING FACTOR VEHICLE 3'] == 'Unsafe Speed') |
    (df['CONTRIBUTING FACTOR VEHICLE 4'] == 'Unsafe Speed') |
    (df['CONTRIBUTING FACTOR VEHICLE 5'] == 'Unsafe Speed')
]

In [19]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

speeding_details = speeding.groupby('BOROUGH').agg({
    'NUMBER OF PERSONS INJURED': 'sum',
    'NUMBER OF PERSONS KILLED': 'sum'
}).reset_index()

speeding_details = speeding_details[speeding_details['BOROUGH'] != 'UnKnOwN']

speeding_details = speeding_details.sort_values(by='NUMBER OF PERSONS INJURED', ascending=False)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Injured", "Killed"),
    shared_xaxes=True
)

# kolumna lewa
fig.add_trace(
    go.Bar(
        x=speeding_details['BOROUGH'],
        y=speeding_details['NUMBER OF PERSONS INJURED'],
        name='Ranni',
        marker_color='#f39c12',
        text=speeding_details['NUMBER OF PERSONS INJURED'],
        textposition='auto'
    ),
    row=1, col=1
)

# kolumna prawa
fig.add_trace(
    go.Bar(
        x=speeding_details['BOROUGH'],
        y=speeding_details['NUMBER OF PERSONS KILLED'],
        name='Zabici',
        marker_color='#c0392b',
        text=speeding_details['NUMBER OF PERSONS KILLED'],
        textposition='auto'
    ),
    row=1, col=2
)

fig.update_layout(
    title_text="Unsafe Speed accidents",
    title_x=0.5,
    title_font_size=18,
    showlegend=False,
    template='plotly_white'
)
fig.update_traces(
    textposition="outside",
)
fig.show()

3.6. Zestawienie ilości wypadków dla każdej z dzielnic

In [20]:
from charts import acc_group_borough
acc_group_borough(df)

3.4. Określenie 3 najczęstszych czynników wypadków z podziałem na dzielnice oraz ogółem dla całego miasta

In [21]:
wypadki_dzielnice = df.groupby(['BOROUGH', 'CONTRIBUTING FACTOR VEHICLE 1']).size().reset_index(name = 'liczba wypadków')
wypadki_dzielnice = wypadki_dzielnice[wypadki_dzielnice['BOROUGH'] != 'UnKnOwN']
wypadki_dzielnice = wypadki_dzielnice[wypadki_dzielnice['CONTRIBUTING FACTOR VEHICLE 1'] != 'Unspecified']

In [22]:
wypadki_dzielnice = wypadki_dzielnice.sort_values(by= ['BOROUGH', 'liczba wypadków'], ascending = [True, False])
#print(wypadki_dzielnice)
top3_per_borough = wypadki_dzielnice.groupby('BOROUGH').head(3)
top3_per_borough

,BOROUGH,CONTRIBUTING FACTOR VEHICLE 1,liczba wypadków
11,BRONX,Driver Inattention/Distraction,32024
21,BRONX,Following Too Closely,10144
32,BRONX,Other Vehicular,8945
73,BROOKLYN,Driver Inattention/Distraction,70310
80,BROOKLYN,Failure to Yield Right-of-Way,26732
83,BROOKLYN,Following Too Closely,18756
135,MANHATTAN,Driver Inattention/Distraction,68450
156,MANHATTAN,Other Vehicular,19473
142,MANHATTAN,Failure to Yield Right-of-Way,14825
197,QUEENS,Driver Inattention/Distraction,85477


In [23]:
import plotly.express as px

fig = px.bar(
    top3_per_borough,
    x="BOROUGH",
    y="liczba wypadków",
    color="CONTRIBUTING FACTOR VEHICLE 1",
    barmode="group",
    text_auto=True
)
fig.update_layout(
    title_text="TOP 3 najczęstszych czynników wypadków z podziałem na dzielnice",
    title_x=0.5,
    title_font_size=18,
    showlegend=True,
    template='plotly_white'
)
fig.update_traces(
    textposition="outside"
)
fig.show()

In [24]:
wypadki_NY = df[(df['BOROUGH'] != 'UnKnOwN') & (df['CONTRIBUTING FACTOR VEHICLE 1'] != 'Unspecified')]
wypadki_NY = wypadki_NY.groupby(['CONTRIBUTING FACTOR VEHICLE 1']).size().reset_index(name='liczba wypadków')

top3_czynniki = wypadki_NY.sort_values(by='liczba wypadków', ascending=False).head(3)

fig = px.bar(
    top3_czynniki,
    x="CONTRIBUTING FACTOR VEHICLE 1",
    y="liczba wypadków",
    text="liczba wypadków",
    color="liczba wypadków",
    color_continuous_scale="Viridis",
    title="3 najczęstsze przyczyny wypadków w NYC",
    labels={"CONTRIBUTING FACTOR VEHICLE 1": "Przyczyna wypadku", "liczba wypadków": "Liczba kolizji"}
)

fig.update_layout(
    title_font_size=16,
    title_x=0.5,
    xaxis_tickangle=0,
    coloraxis_showscale=False,
    template="plotly_white"
)
fig.update_traces(
    textposition="outside"
)
fig.show()

3.5 Określenie jakie pojazdy (typy) najczęściej uczestniczyły w wypadkach

In [25]:
pojazdy_cols = [f'VEHICLE TYPE CODE {i}' for i in range(1, 6)]
wszystkie_pojazdy = df[pojazdy_cols].stack()
niechciane_wpisy = ['UnKnOwN', 'UNKNOWN']
wszystkie_pojazdy = wszystkie_pojazdy[~wszystkie_pojazdy.isin(niechciane_wpisy)]

top_pojazdy = wszystkie_pojazdy.value_counts().head(5).reset_index()
top_pojazdy.columns = ['Typ pojazdu', 'Liczba wypadków']

fig = px.bar(
    top_pojazdy,
    x="Typ pojazdu",
    y="Liczba wypadków",
    text="Liczba wypadków",
    color="Liczba wypadków",
    color_continuous_scale="Cividis",
    title="TOP 5 typów pojazdów najczęściej uczestniczących w wypadkach w NYC",
    labels={"Typ pojazdu": "Typ pojazdu", "Liczba wypadków": "Liczba uczestniczących pojazdów"}
)

fig.update_layout(
    title_font_size=16,
    title_x=0.5,
    xaxis_tickangle=0,
    coloraxis_showscale=False,
    template="plotly_white"
)

fig.update_traces(textposition="outside")

fig.show()

3.7. Zaprezentowanie najczęstszych miejsc wystąpień wypadków

In [ ]:
from map import accident_map
accident_map(df).show()